---
# 📊 ANÁLISIS AVANZADO — Celdas adicionales v8
## Agregar estas celdas al final del notebook transferencias_fraude_v8.ipynb

**Requiere haber corrido todas las celdas anteriores del v8 primero.**

```bash
pip install plotly
```

---
## 📱 CELDA A — Análisis profundo de Push Notification

Push Notification concentra el mayor volumen de fraude (S/838,781 — 84% del total).
No es que sea el método más riesgoso — es el más usado (345k trx).
Aquí entendemos **qué tipo de transacciones PN tienen fraude**.

### 🧠 Hipótesis a validar:
- ¿El fraude en PN se concentra en ciertos bancos destino?
- ¿En ciertos rangos de monto?
- ¿En ciertos días u horas?

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
import numpy as np

# ── Filtrar solo Push Notification ───────────────────────────────────────────
df_pn = df.filter(pl.col('tipo_auth') == 'Push Notification')
df_pn_fraude = df_pn.filter(pl.col('es_fraude') == 1)

print(f'Push Notification — Total trx: {df_pn.shape[0]:,}')
print(f'Push Notification — Fraudes: {df_pn_fraude.shape[0]:,}')
print(f'Push Notification — Monto fraude: S/ {df_pn_fraude["monto"].sum():,.0f}')
print(f'Push Notification — Fraud rate: {df_pn["es_fraude"].mean()*100:.3f}%')

# ── 1. Fraude PN por banco destino ────────────────────────────────────────────
print('\nFraude Push Notification por banco destino:')
pn_banco = (
    df_pn.group_by('banco_destino')
         .agg([
             pl.len().alias('n_trx'),
             pl.col('es_fraude').sum().alias('n_fraudes'),
             (pl.col('es_fraude').mean()*100).round(3).alias('fraud_rate_%'),
             pl.col('monto').filter(pl.col('es_fraude')==1).sum().alias('monto_fraude'),
         ])
         .filter(pl.col('n_fraudes') > 0)
         .sort('monto_fraude', descending=True)
)
print(pn_banco)

# ── 2. Fraude PN por rango de monto ──────────────────────────────────────────
print('\nFraude Push Notification por rango de monto:')
df_pn_pd = df_pn.to_pandas()
bins = [0, 500, 1000, 2000, 3000, 5000, 10000, 20000, 999999]
labels = ['<500','500-1k','1k-2k','2k-3k','3k-5k','5k-10k','10k-20k','>20k']
import pandas as pd
df_pn_pd['rango_monto'] = pd.cut(df_pn_pd['monto'], bins=bins, labels=labels)
rango_pn = df_pn_pd.groupby('rango_monto', observed=True).agg(
    n_trx=('monto','count'),
    n_fraudes=('es_fraude','sum'),
    monto_fraude=('monto', lambda x: x[df_pn_pd.loc[x.index,'es_fraude']==1].sum())
).reset_index()
rango_pn['fraud_rate_%'] = (rango_pn['n_fraudes'] / rango_pn['n_trx'] * 100).round(3)
print(rango_pn.to_string(index=False))

# ── 3. Fraude PN por hora ─────────────────────────────────────────────────────
print('\nFraude Push Notification por hora del día:')
pn_hora = (
    df_pn.group_by('hora')
         .agg([
             pl.len().alias('n_trx'),
             pl.col('es_fraude').sum().alias('n_fraudes'),
             (pl.col('es_fraude').mean()*100).round(3).alias('fraud_rate_%'),
         ])
         .sort('hora')
)
print(pn_hora)

---
## 🔵 CELDA B — Análisis profundo de GMONEY

GMONEY tiene fraud_rate de **3.13%** — 30 veces mayor que Interbank.
Aquí identificamos las cuentas mula de GMONEY y los patrones.

### 🧠 Cómo interpretar:
Si las cuentas mula de GMONEY tienen fraud_rate alto (vs las de Interbank que tenían 0),
significa que el fraude hacia GMONEY está mejor capturado o es más concentrado.

In [ ]:
# ── Filtrar GMONEY ────────────────────────────────────────────────────────────
df_gm = df.filter(pl.col('banco_destino').str.to_uppercase().str.contains('GMONEY'))
df_gm_fraude = df_gm.filter(pl.col('es_fraude') == 1)

print(f'GMONEY — Total trx: {df_gm.shape[0]:,}')
print(f'GMONEY — Fraudes: {df_gm_fraude.shape[0]:,}')
print(f'GMONEY — Fraud rate: {df_gm["es_fraude"].mean()*100:.3f}%')
print(f'GMONEY — Monto fraude total: S/ {df_gm_fraude["monto"].sum():,.0f}')

# ── Top cuentas mula GMONEY ───────────────────────────────────────────────────
print('\nTop cuentas destino GMONEY (posibles cuentas mula):')
mula_gm = (
    df_gm.group_by('cuenta_destino')
         .agg([
             pl.col('cliente_id').n_unique().alias('n_clientes_distintos'),
             pl.len().alias('n_trx_recibidas'),
             pl.col('monto').sum().alias('monto_total_recibido'),
             pl.col('es_fraude').sum().alias('n_fraudes'),
             (pl.col('es_fraude').mean()*100).round(2).alias('fraud_rate_%'),
             pl.col('monto').filter(pl.col('es_fraude')==1).sum().alias('monto_fraude'),
         ])
         .sort('n_clientes_distintos', descending=True)
         .head(15)
)
print(mula_gm)

# ── Perfil del fraude GMONEY — método auth, monto, hora ──────────────────────
print('\nFraude GMONEY por método auth:')
print(
    df_gm.group_by('tipo_auth').agg([
        pl.len().alias('n_trx'),
        pl.col('es_fraude').sum().alias('n_fraudes'),
        (pl.col('es_fraude').mean()*100).round(3).alias('fraud_rate_%'),
        pl.col('monto').filter(pl.col('es_fraude')==1).mean().round(0).alias('monto_prom_fraude'),
    ]).sort('n_fraudes', descending=True)
)

print('\nFraude GMONEY por rango de monto:')
df_gm_pd = df_gm.to_pandas()
df_gm_pd['rango_monto'] = pd.cut(df_gm_pd['monto'], bins=bins, labels=labels)
rango_gm = df_gm_pd.groupby('rango_monto', observed=True).agg(
    n_trx=('monto','count'),
    n_fraudes=('es_fraude','sum'),
).reset_index()
rango_gm['fraud_rate_%'] = (rango_gm['n_fraudes'] / rango_gm['n_trx'] * 100).round(2)
print(rango_gm[rango_gm['n_trx'] > 0].to_string(index=False))

---
## 📋 CELDA C — Tabla maestra de reglas combinadas

Esta tabla responde la pregunta clave: **¿qué combinación de condiciones captura más fraude con menos daño colateral?**

Evaluamos combinaciones de:
- Umbral de monto (S/1,000 / S/3,000 / S/5,000)
- n_trx en últimas 24h (>= 2 / >= 3)
- Banco de riesgo (Interbank + GMONEY + Efectiva)

### 🧠 Métricas de la tabla:
- **recall_%**: % de fraudes que captura la regla
- **precision_%**: de cada 100 alertas, cuántas son fraude real
- **normales_afectados**: clientes buenos que la regla bloquearía
- **monto_fraude_cap**: soles de fraude que se evitarían

In [ ]:
# Bancos de alto riesgo identificados en el análisis
BANCOS_RIESGO_AMPLIO = ['INTERBANK', 'GMONEY', 'EFECTIVA', 'TPP']

df = df.with_columns(
    pl.col('banco_destino').str.to_uppercase()
      .is_in(BANCOS_RIESGO_AMPLIO)
      .cast(pl.Int8).alias('flag_banco_riesgo_amplio')
)

total_f   = int(df['es_fraude'].sum())
total_n   = df.shape[0] - total_f
monto_tf  = df.filter(pl.col('es_fraude')==1)['monto'].sum()

def evaluar_combo(df, nombre, mascara):
    sub  = df.filter(mascara)
    tp   = int(sub.filter(pl.col('es_fraude')==1).shape[0])
    fp   = int(sub.filter(pl.col('es_fraude')==0).shape[0])
    ta   = tp + fp
    mc   = sub.filter(pl.col('es_fraude')==1)['monto'].sum() if tp > 0 else 0
    return {
        'regla':                nombre,
        'alertas_totales':      ta,
        'fraudes_capturados':   tp,
        'recall_%':             round(tp/(total_f+1e-10)*100, 1),
        'precision_%':          round(tp/(ta+1e-10)*100, 2),
        'normales_afectados':   fp,
        'pct_normales_%':       round(fp/(total_n+1e-10)*100, 2),
        'monto_fraude_cap_S/':  round(mc, 0),
        'pct_monto_cap_%':      round(mc/monto_tf*100, 1),
    }

reglas = [
    # ── Solo banco ────────────────────────────────────────────────────────────
    evaluar_combo(df, 'Banco riesgo (solo Interbank)',
        pl.col('flag_dest_riesgo') == 1),
    evaluar_combo(df, 'Banco riesgo amplio (IB+GM+EF+TPP)',
        pl.col('flag_banco_riesgo_amplio') == 1),

    # ── Monto + banco ─────────────────────────────────────────────────────────
    evaluar_combo(df, 'Monto>=1000 + banco amplio',
        (pl.col('monto') >= 1000) & (pl.col('flag_banco_riesgo_amplio') == 1)),
    evaluar_combo(df, 'Monto>=3000 + banco amplio',
        (pl.col('monto') >= 3000) & (pl.col('flag_banco_riesgo_amplio') == 1)),
    evaluar_combo(df, 'Monto>=5000 + banco amplio',
        (pl.col('monto') >= 5000) & (pl.col('flag_banco_riesgo_amplio') == 1)),

    # ── n_trx_24h + banco ─────────────────────────────────────────────────────
    evaluar_combo(df, 'n_trx_24h>=2 + banco amplio',
        (pl.col('n_trx_24h') >= 2) & (pl.col('flag_banco_riesgo_amplio') == 1)),
    evaluar_combo(df, 'n_trx_24h>=3 + banco amplio',
        (pl.col('n_trx_24h') >= 3) & (pl.col('flag_banco_riesgo_amplio') == 1)),

    # ── Monto + n_trx_24h + banco ─────────────────────────────────────────────
    evaluar_combo(df, 'Monto>=1000 + n_trx_24h>=2 + banco amplio',
        (pl.col('monto') >= 1000) & (pl.col('n_trx_24h') >= 2) & (pl.col('flag_banco_riesgo_amplio') == 1)),
    evaluar_combo(df, 'Monto>=3000 + n_trx_24h>=2 + banco amplio',
        (pl.col('monto') >= 3000) & (pl.col('n_trx_24h') >= 2) & (pl.col('flag_banco_riesgo_amplio') == 1)),
    evaluar_combo(df, 'Monto>=5000 + n_trx_24h>=2 + banco amplio',
        (pl.col('monto') >= 5000) & (pl.col('n_trx_24h') >= 2) & (pl.col('flag_banco_riesgo_amplio') == 1)),

    # ── Monto acumulado 24h ───────────────────────────────────────────────────
    evaluar_combo(df, 'Monto_acum_24h>=3000 + banco amplio',
        (pl.col('monto_acum_24h') >= 3000) & (pl.col('flag_banco_riesgo_amplio') == 1)),
    evaluar_combo(df, 'Monto_acum_24h>=5000 + banco amplio',
        (pl.col('monto_acum_24h') >= 5000) & (pl.col('flag_banco_riesgo_amplio') == 1)),
    evaluar_combo(df, 'Monto_acum_24h>=10000 + banco amplio',
        (pl.col('monto_acum_24h') >= 10000) & (pl.col('flag_banco_riesgo_amplio') == 1)),

    # ── Auth débil + banco ────────────────────────────────────────────────────
    evaluar_combo(df, 'Auth debil (SMS/Email) + banco amplio',
        (pl.col('flag_auth_debil') == 1) & (pl.col('flag_banco_riesgo_amplio') == 1)),

    # ── Combinaciones fuertes ─────────────────────────────────────────────────
    evaluar_combo(df, 'Monto>=1000 + n_trx_24h>=2 + auth_debil + banco amplio',
        (pl.col('monto') >= 1000) & (pl.col('n_trx_24h') >= 2) &
        (pl.col('flag_auth_debil') == 1) & (pl.col('flag_banco_riesgo_amplio') == 1)),
    evaluar_combo(df, 'OR general: monto>=3k OR (n_trx_24h>=2+banco) OR mula',
        (pl.col('monto') >= 3000) |
        ((pl.col('n_trx_24h') >= 2) & (pl.col('flag_banco_riesgo_amplio') == 1)) |
        (pl.col('flag_mula_destino') == 1)),
]

tabla_reglas = pl.DataFrame(reglas)
print('TABLA MAESTRA DE REGLAS COMBINADAS:\n')
print('Referencia → Total fraudes:', total_f, '| Monto total fraude: S/',round(monto_tf,0))
print()
tabla_reglas

---
## 🔗 CELDA D — Asociación de variables categóricas con fraude

Usamos **Chi-cuadrado** y **Cramér's V** — el estándar estadístico para medir asociación entre variables categóricas.

### 🧠 Cómo interpretar Cramér's V:

| V | Asociación |
|---|---|
| < 0.10 | Débil / sin asociación práctica |
| 0.10 - 0.20 | Moderada |
| 0.20 - 0.40 | Fuerte |
| > 0.40 | Muy fuerte |

El p-value del Chi-cuadrado te dice si la asociación es estadísticamente significativa (p < 0.05 = sí lo es).

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

df_pd = df.to_pandas()

categoricas = ['banco_destino', 'tipo_auth', 'segmento', 'dia_semana',
               'flag_nocturno', 'flag_fin_semana', 'flag_dest_riesgo',
               'es_primera_vez_destino', 'flag_auth_debil',
               'flag_banco_riesgo_amplio']

resultados_chi2 = []
for col in categoricas:
    if col not in df_pd.columns:
        continue
    tabla = pd.crosstab(df_pd[col], df_pd['es_fraude'])
    if tabla.shape[0] < 2:
        continue
    chi2, p, dof, _ = chi2_contingency(tabla)
    n = tabla.values.sum()
    k = min(tabla.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * k)) if k > 0 else 0
    resultados_chi2.append({
        'variable':    col,
        'cramers_v':   round(cramers_v, 4),
        'chi2':        round(chi2, 2),
        'p_value':     round(p, 6),
        'significativa': 'SÍ ✓' if p < 0.05 else 'NO ✗',
        'fuerza':      'Muy fuerte' if cramers_v > 0.4 else
                       'Fuerte' if cramers_v > 0.2 else
                       'Moderada' if cramers_v > 0.1 else 'Débil',
    })

chi2_df = pd.DataFrame(resultados_chi2).sort_values('cramers_v', ascending=False)
print('ASOCIACIÓN DE VARIABLES CATEGÓRICAS CON FRAUDE (Cramér V + Chi2):\n')
print(chi2_df.to_string(index=False))

---
## 📊 CELDA E — Gráficos: Panorama general del fraude

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

df_pd = df.to_pandas()
df_fraude_pd = df_pd[df_pd['es_fraude'] == 1]

# ── Gráfico 1: Distribución del indicador de fraude ───────────────────────────
ind_df = df_pd.groupby('indicador_fraude').agg(
    n_trx=('monto','count'),
    monto_total=('monto','sum')
).reset_index()
ind_df['etiqueta'] = ind_df['indicador_fraude'].map({
    'F':'Fraude (F)', 'N':'Normal (N)', 'G':'Good (G)',
    'D':'Descarte (D)', 'P':'Pendiente (P)'
})

fig1 = make_subplots(rows=1, cols=2,
    subplot_titles=['Distribución por N° de transacciones', 'Distribución por Monto (S/)'])
fig1.add_trace(go.Bar(x=ind_df['etiqueta'], y=ind_df['n_trx'],
    marker_color=['#e74c3c','#2ecc71','#3498db','#f39c12','#9b59b6'],
    text=ind_df['n_trx'].apply(lambda x: f'{x:,}'), textposition='outside',
    name='N° Trx'), row=1, col=1)
fig1.add_trace(go.Bar(x=ind_df['etiqueta'], y=ind_df['monto_total'],
    marker_color=['#e74c3c','#2ecc71','#3498db','#f39c12','#9b59b6'],
    text=ind_df['monto_total'].apply(lambda x: f'S/{x/1e6:.1f}M'), textposition='outside',
    name='Monto'), row=1, col=2)
fig1.update_layout(title='Distribución del Indicador de Fraude — Marzo',
    height=450, showlegend=False)
fig1.show()

# ── Gráfico 2: Fraud rate por banco destino ───────────────────────────────────
banco_df = df_pd.groupby('banco_destino').agg(
    n_trx=('monto','count'),
    n_fraudes=('es_fraude','sum'),
    monto_fraude=('monto', lambda x: x[df_pd.loc[x.index,'es_fraude']==1].sum())
).reset_index()
banco_df['fraud_rate_%'] = banco_df['n_fraudes'] / banco_df['n_trx'] * 100
banco_df = banco_df[banco_df['n_fraudes'] > 0].sort_values('monto_fraude', ascending=False).head(12)

fig2 = make_subplots(rows=1, cols=2,
    subplot_titles=['Monto de fraude por banco (S/)', 'Fraud rate % por banco'])
colores = ['#e74c3c' if b in ['Interbank','GMONEY S.A','Efectiva','TPP (Tarjetas Peruanas Prepago…']
           else '#3498db' for b in banco_df['banco_destino']]
fig2.add_trace(go.Bar(x=banco_df['banco_destino'], y=banco_df['monto_fraude'],
    marker_color=colores, name='Monto fraude',
    text=banco_df['monto_fraude'].apply(lambda x: f'S/{x:,.0f}'), textposition='outside'),
    row=1, col=1)
fig2.add_trace(go.Bar(x=banco_df['banco_destino'], y=banco_df['fraud_rate_%'],
    marker_color=colores, name='Fraud rate %',
    text=banco_df['fraud_rate_%'].apply(lambda x: f'{x:.2f}%'), textposition='outside'),
    row=1, col=2)
fig2.update_layout(title='Fraude por Banco Destino — Rojo = bancos de alto riesgo',
    height=500, showlegend=False)
fig2.update_xaxes(tickangle=45)
fig2.show()

---
## 📊 CELDA F — Gráficos: Patrones temporales y de monto

In [ ]:
# ── Gráfico 3: Fraud rate por hora del día ────────────────────────────────────
hora_df = df_pd.groupby('hora').agg(
    n_trx=('monto','count'),
    n_fraudes=('es_fraude','sum')
).reset_index()
hora_df['fraud_rate_%'] = hora_df['n_fraudes'] / hora_df['n_trx'] * 100

fig3 = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['Volumen de transacciones por hora', 'Fraud rate % por hora'])
fig3.add_trace(go.Bar(x=hora_df['hora'], y=hora_df['n_trx'],
    marker_color='#3498db', name='N° Trx'), row=1, col=1)
fig3.add_trace(go.Scatter(x=hora_df['hora'], y=hora_df['fraud_rate_%'],
    mode='lines+markers', line=dict(color='#e74c3c', width=2),
    marker=dict(size=8), name='Fraud rate %'), row=2, col=1)
fig3.update_xaxes(title_text='Hora del día', tickmode='linear', row=2, col=1)
fig3.update_layout(title='Patrón temporal del fraude — Por hora del día',
    height=500, showlegend=False)
fig3.show()

# ── Gráfico 4: Distribución de montos — Fraude vs Normal ──────────────────────
df_muestra_normal = df_pd[df_pd['es_fraude']==0].sample(min(2000, len(df_pd[df_pd['es_fraude']==0])), random_state=42)
df_muestra_fraude = df_pd[df_pd['es_fraude']==1]

fig4 = go.Figure()
fig4.add_trace(go.Histogram(
    x=df_muestra_normal['monto'].clip(upper=20000),
    nbinsx=50, name='Normal', opacity=0.6,
    marker_color='#3498db'))
fig4.add_trace(go.Histogram(
    x=df_muestra_fraude['monto'].clip(upper=20000),
    nbinsx=50, name='Fraude', opacity=0.8,
    marker_color='#e74c3c'))
fig4.update_layout(
    title='Distribución de montos — Fraude vs Normal (hasta S/20,000)',
    xaxis_title='Monto (S/)', yaxis_title='Frecuencia',
    barmode='overlay', height=400)
fig4.show()

# ── Gráfico 5: Fraud rate por día de semana ───────────────────────────────────
dia_df = df_pd.groupby('dia_semana').agg(
    n_trx=('monto','count'),
    n_fraudes=('es_fraude','sum')
).reset_index()
dia_df['fraud_rate_%'] = dia_df['n_fraudes'] / dia_df['n_trx'] * 100
dia_df['dia_nombre'] = dia_df['dia_semana'].map({
    0:'Lun', 1:'Mar', 2:'Mié', 3:'Jue', 4:'Vie', 5:'Sáb', 6:'Dom'
})

fig5 = make_subplots(rows=1, cols=2,
    subplot_titles=['N° transacciones por día', 'Fraud rate % por día'])
colores_dia = ['#e74c3c' if d >= 5 else '#3498db' for d in dia_df['dia_semana']]
fig5.add_trace(go.Bar(x=dia_df['dia_nombre'], y=dia_df['n_trx'],
    marker_color=colores_dia, name='N° Trx'), row=1, col=1)
fig5.add_trace(go.Bar(x=dia_df['dia_nombre'], y=dia_df['fraud_rate_%'],
    marker_color=colores_dia,
    text=dia_df['fraud_rate_%'].apply(lambda x: f'{x:.3f}%'), textposition='outside',
    name='Fraud rate'), row=1, col=2)
fig5.update_layout(title='Patrón por día de la semana — Rojo = fin de semana',
    height=400, showlegend=False)
fig5.show()

---
## 📊 CELDA G — Gráficos: Autenticación y ventanas de velocidad

In [ ]:
# ── Gráfico 6: Fraude por método de autenticación ────────────────────────────
auth_df = df_pd.groupby('tipo_auth').agg(
    n_trx=('monto','count'),
    n_fraudes=('es_fraude','sum'),
    monto_fraude=('monto', lambda x: x[df_pd.loc[x.index,'es_fraude']==1].sum())
).reset_index()
auth_df['fraud_rate_%'] = auth_df['n_fraudes'] / auth_df['n_trx'] * 100
auth_df = auth_df.sort_values('fraud_rate_%', ascending=False)

colores_auth = ['#e74c3c' if a in ['SMS','Email'] else
                '#2ecc71' if a in ['Touch ID','Face ID'] else
                '#f39c12' for a in auth_df['tipo_auth']]

fig6 = make_subplots(rows=1, cols=2,
    subplot_titles=['Fraud rate % por método auth', 'Monto fraude por método auth'])
fig6.add_trace(go.Bar(x=auth_df['tipo_auth'], y=auth_df['fraud_rate_%'],
    marker_color=colores_auth,
    text=auth_df['fraud_rate_%'].apply(lambda x: f'{x:.3f}%'), textposition='outside',
    name='Fraud rate'), row=1, col=1)
fig6.add_trace(go.Bar(x=auth_df['tipo_auth'], y=auth_df['monto_fraude'],
    marker_color=colores_auth,
    text=auth_df['monto_fraude'].apply(lambda x: f'S/{x:,.0f}'), textposition='outside',
    name='Monto fraude'), row=1, col=2)
fig6.update_layout(
    title='Fraude por método de autenticación — Rojo=débil, Verde=biometría',
    height=450, showlegend=False)
fig6.show()

# ── Gráfico 7: Comparación ventanas rolling — Fraude vs Normal ────────────────
ventana_comp = df_pd.groupby('es_fraude').agg(
    n_trx_3min=('n_trx_3min','mean'),
    n_trx_5min=('n_trx_5min','mean'),
    n_trx_1h=('n_trx_1h','mean'),
    n_trx_12h=('n_trx_12h','mean'),
    n_trx_24h=('n_trx_24h','mean'),
).reset_index()
ventana_comp['tipo'] = ventana_comp['es_fraude'].map({0:'Normal', 1:'Fraude'})

ventanas_cols = ['n_trx_3min','n_trx_5min','n_trx_1h','n_trx_12h','n_trx_24h']
etiquetas = ['3 min','5 min','1 hora','12 horas','24 horas']

fig7 = go.Figure()
for _, row in ventana_comp.iterrows():
    color = '#e74c3c' if row['tipo'] == 'Fraude' else '#3498db'
    fig7.add_trace(go.Scatter(
        x=etiquetas,
        y=[row[c] for c in ventanas_cols],
        mode='lines+markers',
        name=row['tipo'],
        line=dict(color=color, width=3),
        marker=dict(size=10)
    ))
fig7.update_layout(
    title='Promedio de transacciones por ventana de tiempo — Fraude vs Normal',
    xaxis_title='Ventana de tiempo',
    yaxis_title='N° trx promedio',
    height=400
)
fig7.show()

# ── Gráfico 8: Monto acumulado por ventana ────────────────────────────────────
monto_comp = df_pd.groupby('es_fraude').agg(
    monto_3min=('monto_acum_3min','mean'),
    monto_5min=('monto_acum_5min','mean'),
    monto_1h=('monto_acum_1h','mean'),
    monto_12h=('monto_acum_12h','mean'),
    monto_24h=('monto_acum_24h','mean'),
).reset_index()
monto_comp['tipo'] = monto_comp['es_fraude'].map({0:'Normal', 1:'Fraude'})

monto_cols = ['monto_3min','monto_5min','monto_1h','monto_12h','monto_24h']

fig8 = go.Figure()
for _, row in monto_comp.iterrows():
    color = '#e74c3c' if row['tipo'] == 'Fraude' else '#3498db'
    fig8.add_trace(go.Scatter(
        x=etiquetas,
        y=[row[c] for c in monto_cols],
        mode='lines+markers',
        name=row['tipo'],
        line=dict(color=color, width=3),
        marker=dict(size=10)
    ))
fig8.update_layout(
    title='Monto acumulado promedio por ventana de tiempo — Fraude vs Normal',
    xaxis_title='Ventana de tiempo',
    yaxis_title='Monto acumulado promedio (S/)',
    height=400
)
fig8.show()

---
## 📊 CELDA H — Gráficos: Tabla maestra de reglas y discriminación

In [ ]:
# ── Gráfico 9: Lift por feature ───────────────────────────────────────────────
disc_pd = disc_df.to_pandas() if hasattr(disc_df, 'to_pandas') else disc_df
disc_pd = disc_pd.sort_values('lift', ascending=True)
colores_lift = ['#e74c3c' if l > 2 else '#f39c12' if l > 1.2 else '#95a5a6'
                for l in disc_pd['lift']]

fig9 = go.Figure(go.Bar(
    x=disc_pd['lift'], y=disc_pd['feature'],
    orientation='h',
    marker_color=colores_lift,
    text=disc_pd['lift'].apply(lambda x: f'{x:.2f}x'),
    textposition='outside'
))
fig9.add_vline(x=1, line_dash='dash', line_color='gray',
    annotation_text='Lift=1 (sin discriminación)')
fig9.update_layout(
    title='Poder discriminante por feature (Lift) — Rojo=fuerte, Naranja=moderado',
    xaxis_title='Lift', height=500
)
fig9.show()

# ── Gráfico 10: Tabla de reglas — Recall vs Normales afectados ────────────────
tabla_pd = tabla_reglas.to_pandas()

fig10 = px.scatter(
    tabla_pd,
    x='pct_normales_%',
    y='recall_%',
    size='monto_fraude_cap_S/',
    color='precision_%',
    hover_name='regla',
    color_continuous_scale='RdYlGn',
    text='regla',
    title='Reglas de control: Recall vs % Normales afectados<br>(tamaño = monto fraude capturado, color = precisión)',
    labels={
        'pct_normales_%': '% Normales afectados (menor es mejor →)',
        'recall_%': 'Recall % (mayor es mejor ↑)'
    }
)
fig10.update_traces(textposition='top center', textfont_size=8)
fig10.update_layout(height=600)
fig10.show()

print('\n💡 La regla IDEAL está en la esquina SUPERIOR IZQUIERDA del gráfico:')
print('   → Recall alto (captura mucho fraude) + pocos normales afectados')

---
## 💾 CELDA I — Exportar tabla maestra de reglas

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tabla_reglas.write_csv(OUTPUT_DIR / 'tabla_maestra_reglas.csv')
chi2_df.to_csv(OUTPUT_DIR / 'asociacion_categoricas_chi2.csv', index=False)
mula_gm.write_csv(OUTPUT_DIR / 'cuentas_mula_gmoney.csv')

print(f'✓ Exportado a: {OUTPUT_DIR}')
print('  - tabla_maestra_reglas.csv        (todas las combinaciones de reglas)')
print('  - asociacion_categoricas_chi2.csv (CramérV + Chi2 por variable)')
print('  - cuentas_mula_gmoney.csv         (top cuentas mula GMONEY)')